# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a template for loading and exploring a dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import json
import warnings
warnings.filterwarnings('ignore')

# Define the dataset URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
# The .metadata is an object, use the .to_json() method for a dict-like interface
metadata = dataset.metadata.to_json()
print(f"Title: {metadata.get('name')}\nDescription: {metadata.get('description')}")
print(f"License: {metadata.get('license')}")
print(f"Published: {metadata.get('datePublished')}")

## 2. Data Overview
Review available record sets, fields, and their IDs.

The Croissant schema allows inspection of record sets (tables) and their fields and columns. We extract and list available `@id`s and metadata here.

In [ ]:
# Get all record sets in the dataset
record_sets = dataset.metadata.record_sets
print(f"Number of record sets: {len(record_sets)}\n")
for rs in record_sets:
    print(f"RecordSet: {rs['@id']}")
    print(f"  Name: {rs.get('name', 'N/A')}")
    print(f"  Description: {rs.get('description', 'N/A')}")
    # List fields and columns, using their `@id`
    fields = rs.get('fields', [])
    if fields:
        print("  Fields:")
        for field in fields:
            print(f"    Field @id: {field['@id']}  Name: {field.get('name', 'N/A')}")
            if 'columns' in field:
                for column in field['columns']:
                    print(f"      Column @id: {column['@id']}  Name: {column.get('name', 'N/A')}")
    else:
        print("  No fields listed.")
    print()

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. Use the record set and field `@id`s from the overview.

In [ ]:
# Example: extract the main record set (usually the protein data table)
# The actual `@id` may need adaptation based on the printed overview above.

# You can update this variable to another RecordSet ID if needed after inspecting above
main_record_set_id = None
for rs in dataset.metadata.record_sets:
    # Heuristically pick a protein/EV table if present
    if 'protein' in rs.get('name', '').lower() or 'ev' in rs.get('name', '').lower() or 'main' in rs.get('name', '').lower():
        main_record_set_id = rs['@id']
        break
if main_record_set_id is None and len(dataset.metadata.record_sets) > 0:
    main_record_set_id = dataset.metadata.record_sets[0]['@id']  # fallback: pick first RecordSet
print(f"Using record set: {main_record_set_id}\n")

all_record_set_ids = [rs['@id'] for rs in dataset.metadata.record_sets]
print(f"All record set IDs in this dataset: {all_record_set_ids}")

dataframes = {}
for record_set_id in all_record_set_ids:
    print(f"Loading records from RecordSet: {record_set_id}")
    records = list(dataset.records(record_set=record_set_id))
    df = pd.DataFrame(records)
    dataframes[record_set_id] = df
    print(f"Loaded {len(df)} records. Columns: {df.columns.tolist()}\n")

if main_record_set_id in dataframes:
    print("Sample of the main record set:")
    display(dataframes[main_record_set_id].head())

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data. This section should include operations like removing outliers, transforming data distributions, or grouping data by key attributes to prepare it for further analysis.

In [ ]:
# Choose numeric and group fields from the DataFrame columns, using their Croissant @id where possible.

df = dataframes[main_record_set_id]
numeric_field_candidates = [col for col in df.columns if df[col].dtype.kind in 'fi' and not df[col].isnull().all()]
print(f"Numeric fields: {numeric_field_candidates}")

# Example numeric field, fallback to first available
numeric_field = numeric_field_candidates[0] if numeric_field_candidates else df.columns[0]
print(f"Using numeric field: {numeric_field}")

# Example group-by field, pick a categorical/text column, if available
category_fields = [col for col in df.columns if df[col].dtype == object and df[col].nunique() < len(df)//2]
group_field = category_fields[0] if category_fields else df.columns[0]
print(f"Using group field: {group_field}\n")

# Filtering step (arbitrary threshold for demonstration)
if pd.api.types.is_numeric_dtype(df[numeric_field]):
    threshold = df[numeric_field].quantile(0.75)  # top quartile
    filtered_df = df[df[numeric_field] > threshold].copy()
    print(f"Filtered records with {numeric_field} > {threshold} (top quartile)")
    display(filtered_df.head())

    # Normalization
    mean = filtered_df[numeric_field].mean()
    std = filtered_df[numeric_field].std()
    filtered_df[f"{numeric_field}_normalized"] = (filtered_df[numeric_field] - mean) / std
    print(f"Normalized {numeric_field} for filtered records:")
    display(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())

    # Group by group_field (if appropriate)
    if group_field in filtered_df.columns:
        grouped_df = filtered_df.groupby(group_field)[numeric_field].mean().reset_index()
        print(f"Grouped data by {group_field} (mean {numeric_field}):")
        display(grouped_df.head())
else:
    print("No numeric field suitable for analysis.")

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if pd.api.types.is_numeric_dtype(df[numeric_field]):
    plt.figure(figsize=(8,5))
    sns.histplot(df[numeric_field].dropna(), bins=30, kde=True)
    plt.title(f'Distribution of {numeric_field}')
    plt.xlabel(numeric_field)
    plt.ylabel('Count')
    plt.show()

    # If enough data and group field, make a boxplot
    n_groups = df[group_field].nunique()
    if n_groups > 1 and n_groups < 50:
        plt.figure(figsize=(10,6))
        sns.boxplot(x=group_field, y=numeric_field, data=df)
        plt.title(f'{numeric_field} by {group_field}')
        plt.xticks(rotation=45)
        plt.show()

## 6. Conclusion
Summarize key findings and observations from the dataset exploration.

- We loaded the "Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells" dataset using the `mlcroissant` library and inspected its metadata, structure, and main tables by referencing all entities by their Croissant `@id`.
- We performed initial exploratory analysis, including filtering and normalization of numeric fields and visualization of data distributions.
- This approach allows robust, reproducible access to FAIR datasets and prepares the data for downstream ML and statistical modeling workflows.